# Primeira parte do Trabalho Prático de BD2

**Aluno:** Atos Medina Araújo  
**Curso:** Análise e Desenvolvimento de Sistemas  
**Disciplina:** Banco de Dados II  
**Tema do PI3:** Reações Químicas, Moléculas e Átomos


In [ ]:
import sqlite3
from sqlite3 import Error

# Função de alteração (insert, update, delete...)
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print(f"Query executada com sucesso.")
        if cursor.rowcount != -1:
            print(f"{cursor.rowcount} linha(s) afetadas")
    except Error as e:
        print(f"Erro: '{e}'")

# Função de leitura de dados (select)
def execute_read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as e:
        print(f"Erro: '{e}'")

# Conexão em memória com SQLite
conn = sqlite3.connect(':memory:')
print("Conexão com SQLite estabelecida!")

## Suas tabelas de PI3

Criação das tabelas baseadas no Diagrama Entidade-Relacionamento de Reações Químicas.

cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

schema_sql = """
CREATE TABLE ReacaoQuimica (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    tipo TEXT NOT NULL,
    equacao_balanceada TEXT NOT NULL,
    instrucoes TEXT
);

CREATE TABLE Molecula (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    formula TEXT NOT NULL
);

CREATE TABLE Atomo (
    numero_atomico INTEGER PRIMARY KEY,
    simbolo TEXT NOT NULL,
    nome TEXT NOT NULL
);

CREATE TABLE Reagente (
    id_reacao INTEGER,
    id_molecula INTEGER,
    quantidade REAL NOT NULL,
    PRIMARY KEY (id_reacao, id_molecula),
    FOREIGN KEY (id_reacao) REFERENCES ReacaoQuimica(id),
    FOREIGN KEY (id_molecula) REFERENCES Molecula(id)
);

CREATE TABLE Produto (
    id_reacao INTEGER,
    id_molecula INTEGER,
    quantidade REAL NOT NULL,
    PRIMARY KEY (id_reacao, id_molecula),
    FOREIGN KEY (id_reacao) REFERENCES ReacaoQuimica(id),
    FOREIGN KEY (id_molecula) REFERENCES Molecula(id)
);

CREATE TABLE Composicao (
    id_molecula INTEGER,
    numero_atomico INTEGER,
    quantidade INTEGER NOT NULL,
    PRIMARY KEY (id_molecula, numero_atomico),
    FOREIGN KEY (id_molecula) REFERENCES Molecula(id),
    FOREIGN KEY (numero_atomico) REFERENCES Atomo(numero_atomico)
);

-- Inserção de Dados
INSERT INTO ReacaoQuimica (nome, tipo, equacao_balanceada, instrucoes)
VALUES ('Síntese da Água', 'Síntese', '2H2 + O2 -> 2H2O', 'Misturar hidrogênio e oxigênio com faísca elétrica'),
       ('Combustão do Metano', 'Combustão', 'CH4 + 2O2 -> CO2 + 2H2O', 'Combustão na presença de oxigênio');

INSERT INTO Molecula (nome, formula)
VALUES ('Gás Hidrogênio', 'H2'), ('Gás Oxigênio', 'O2'), ('Água', 'H2O'), ('Metano', 'CH4'), ('Dióxido de Carbono', 'CO2');

INSERT INTO Atomo (numero_atomico, simbolo, nome)
VALUES (1, 'H', 'Hidrogênio'), (6, 'C', 'Carbono'), (8, 'O', 'Oxigênio');

INSERT INTO Composicao VALUES (1, 1, 2);
INSERT INTO Composicao VALUES (2, 8, 2);
INSERT INTO Composicao VALUES (3, 1, 2);
INSERT INTO Composicao VALUES (3, 8, 1);

INSERT INTO Reagente VALUES (1, 1, 2.0);
INSERT INTO Reagente VALUES (1, 2, 1.0);
INSERT INTO Produto VALUES (1, 3, 2.0);
"""

cursor.executescript(schema_sql)
conn.commit()
print("Tabelas do PI3 criadas com sucesso!")

## Atividade 01
Crie duas consultas relevantes para sua aplicação em PI3 que tenham, pelo menos, uma função de agregação em cada. Para cada consulta, descreva a ação esperada (ex.: contar participantes em um evento) e justifique o contexto na aplicação de PI3 (ex.: funcionalidade do back-end para checar se já atingiu o limite de vagas do evento).




CONSULTA 1

* **Objetivo da consulta (contexto de PI3):** Contar a quantidade total de moléculas reagentes cadastradas para cada reação química no banco de dados.
* **Descrição da ação esperada ao rodar a consulta:** Retorna o ID da reação e a contagem total de reagentes associados utilizando a função de agregação COUNT.

In [ ]:
## ESCREVA AQUI O SELECT 1


query_1 = """
SELECT id_reacao, COUNT(id_molecula) AS total_reagentes
FROM Reagente
GROUP BY id_reacao;
"""

resultado_1 = execute_read_query(conn, query_1)
print("Resultado Consulta 1:", resultado_1)



CONSULTA 2

* **Objetivo da consulta (contexto de PI3):** Calcular a quantidade média de átomos por molécula no sistema para análise estequiométrica no back-end.
* **Descrição da ação esperada ao rodar a consulta:** Retorna a média da soma de átomos que compõem cada molécula através da função de agregação AVG.

In [ ]:
## ESCREVA AQUI A CONSULTA 2
query_2 = """
SELECT AVG(total_atomos) AS media_atomos_por_molecula
FROM (
    SELECT SUM(quantidade) AS total_atomos
    FROM Composicao
    GROUP BY id_molecula
);
"""

resultado_2 = execute_read_query(conn, query_2)
print("Resultado Consulta 2:", resultado_2)

## Atividade 02

Crie uma consulta relevante para sua aplicação em PI3 que envolva três entidades distintas. Descreva a ação esperada e justifique o contexto na aplicação de PI3.

Atividade 02

Crie uma consulta relevante para sua aplicação em PI3 que envolva três entidades distintas. Descreva a ação esperada e justifique o contexto na aplicação de PI3.

-- EDITE ESSA CÉLULA --

CONSULTA COM 3 ENTIDADES

* **Objetivo da consulta (contexto de PI3):** Listar quais átomos formam uma determinada molécula e qual a quantidade de cada um, relacionando as tabelas Molecula, Composicao e Atomo.
* **Descrição da ação esperada ao rodar a consulta:** Retorna o nome da molécula, sua fórmula, o nome do átomo e a quantidade do átomo presente na molécula através de um JOIN entre 3 entidades.

In [ ]:
## ESCREVA AQUI A CONSULTA COM 3 ENTIDADES

query_3_entidades = """
SELECT m.nome AS molecula, m.formula, a.nome AS atomo, c.quantidade
FROM Molecula m
JOIN Composicao c ON m.id = c.id_molecula
JOIN Atomo a ON c.numero_atomico = a.numero_atomico;
"""

resultado_3 = execute_read_query(conn, query_3_entidades)
print("Resultado Consulta 3 Entidades:", resultado_3)

## Atividade 03

Crie uma view relevante para sua aplicação em PI3. Descreva a ação esperada na consulta dentro de sua view e, ao justificar o contexto, informe também por que a ação é tão recorrente no seu PI3 a ponto de precisar de uma view.

Atividade 03

Crie uma view relevante para sua aplicação em PI3. Descreva a ação esperada na consulta dentro de sua view e, ao justificar o contexto, informe também por que a ação é tão recorrente no seu PI3 a ponto de precisar de uma view.

-- EDITE ESSA CÉLULA --

* **Justificativa para que a consulta seja uma View (Contexto de PI3):** A listagem completa das reações químicas com seus respectivos produtos e fórmulas é a consulta mais acessada na interface principal. Encapsulá-la em uma View evita a reescrita constante de JOINs complexos.
* **Descrição da ação esperada ao rodar a consulta:** Cria uma visualização unificada das reações químicas integradas com os seus produtos finais.

In [ ]:
## ESCREVA ABAIXO A VIEW
sql_create_view = """
CREATE VIEW IF NOT EXISTS vw_detalhes_produtos AS
SELECT r.nome AS reacao, r.tipo, m.nome AS produto, m.formula, p.quantidade
FROM ReacaoQuimica r
JOIN Produto p ON r.id = p.id_reacao
JOIN Molecula m ON p.id_molecula = m.id;
"""

execute_query(conn, sql_create_view)

## ESCREVA ABAIXO O SELECT QUE RETORNA TODOS OS ELEMENTOS DA VIEW
query_view = "SELECT * FROM vw_detalhes_produtos;"
resultado_view = execute_read_query(conn, query_view)
print("Dados da View:", resultado_view)

## Atividade 04

Crie um procedure relevante para sua aplicação em PI3. Descreva a ação esperada ao executar a procedure e, ao justificar o contexto, explique por que essa ação é tão recorrente no seu PI3 a ponto de precisar de uma procedure.




* **Justificativa para que a operação seja um Procedure (Contexto de PI3):** Alterar as instruções de preparo de uma reação química é uma ação administrativa frequente. O procedure padroniza a atualização e evita erros de comando manual.
* **Descrição da ação esperada ao executar o procedure:** Atualiza as instruções da reação informada pelo parâmetro de ID.

In [ ]:
## ESCREVA ABAIXO O PROCEDURE
def sp_atualizar_instrucoes(connection, p_id_reacao, p_novas_instrucoes):
    query = f"""
    UPDATE ReacaoQuimica
    SET instrucoes = '{p_novas_instrucoes}'
    WHERE id = {p_id_reacao};
    """
    execute_query(connection, query)

## ESCREVA ABAIXO UMA EXECUÇÃO DA PROCEDURE
sp_atualizar_instrucoes(conn, 1, 'Utilizar catalisador de platina e faísca elétrica.')

print(execute_read_query(conn, "SELECT id, nome, instrucoes FROM ReacaoQuimica WHERE id = 1;"))

## Atividade 05

Crie uma function relevante para sua aplicação em PI3.
Descreva o valor esperado como retorno ao executar a function e, ao justificar o contexto, explique por que é importante ter essa lógica encapsulada para uso recorrente no seu PI3.



* **Justificativa para que a operação seja uma Function (Contexto de PI3):** O cálculo da quantidade total de átomos em uma molécula é uma regra de negócio necessária em várias rotinas do sistema.
* **Descrição da operação realizada e valor retornado pela Function:** Soma as quantidades de átomos que compõem a molécula indicada e retorna o número total acumulado.

In [ ]:
## ESCREVA ABAIXO A FUNCTION
def fn_total_atomos_molecula(id_molecula):
    cursor = conn.cursor()
    cursor.execute("SELECT SUM(quantidade) FROM Composicao WHERE id_molecula = ?", (id_molecula,))
    row = cursor.fetchone()
    return row[0] if row[0] is not None else 0

conn.create_function("fn_total_atomos_molecula", 1, fn_total_atomos_molecula)

## ESCREVA ABAIXO UMA CONSULTA QUE INCLUI A FUNCTION
query_fn = "SELECT nome, formula, fn_total_atomos_molecula(id) AS total_atomos FROM Molecula;"
resultado_fn = execute_read_query(conn, query_fn)
print("Resultado da Function:", resultado_fn)

## Atividade 06

Crie um trigger relevante para sua aplicação em PI3.
Descreva o evento que aciona o trigger e, ao justificar o contexto, explique por que essa automação é necessária para manter a integridade ou eficiência do seu PI3.



* **Justificativa para que a operação seja um Trigger (Contexto de PI3):** Garantir a integridade do banco de dados impedindo o cadastro acidental de moléculas sem fórmula química.
* **Descrição do evento que dispara o Trigger e das operações que ele executa:** Disparado BEFORE INSERT na tabela Molecula. Se a fórmula estiver vazia, o Trigger interrompe e cancela a inserção.

In [ ]:
## ESCREVA ABAIXO O TRIGGER
sql_trigger = """
CREATE TRIGGER IF NOT EXISTS trg_valida_formula_molecula
BEFORE INSERT ON Molecula
FOR EACH ROW
BEGIN
    SELECT
        CASE
            WHEN NEW.formula IS NULL OR NEW.formula = '' THEN
                RAISE(ABORT, 'Erro: A molécula precisa ter uma fórmula válida.')
        END;
END;
"""

execute_query(conn, sql_trigger)

## ESCREVA ABAIXO UM COMANDO SQL QUE DISPARA O TRIGGER
execute_query(conn, "INSERT INTO Molecula (nome, formula) VALUES ('Amônia', 'NH3');")

print(execute_read_query(conn, "SELECT * FROM Molecula;"))